# Évaluation du modèle de sentiment

Fait passer les 30 avis annotés dans l'API, construit la matrice de confusion 3×3, puis compare la conversion actuelle (seuils 2.8/3.6) à une conversion alternative à seuils symétriques (2.5/3.5).

**Prérequis** : l'API doit déjà tourner sur `127.0.0.1:8000` (`docker compose up` ou `uvicorn app.main:app`).

In [1]:
import csv
from pathlib import Path

import pandas as pd
import requests

API_URL = "http://127.0.0.1:8000/predict"
CSV_PATH = Path.cwd().resolve().parents[1] / "data" / "avis_annotes.csv"

CLASSES = ("négatif", "neutre", "positif")
ETOILES = ("1 star", "2 stars", "3 stars", "4 stars", "5 stars")

# Conversion actuelle de l'API (cf. app/inference.py) : seuils asymétriques
SEUIL_NEG_ACTUEL, SEUIL_POS_ACTUEL = 2.8, 3.6

# Deuxième conversion à tester : seuils symétriques autour de 3, écartée par l'équipe
SEUIL_NEG_ALT, SEUIL_POS_ALT = 2.5, 3.5


def note_attendue(scores: dict) -> float:
    return sum((i + 1) * scores[e] for i, e in enumerate(ETOILES))


def convertir(note: float, seuil_neg: float, seuil_pos: float) -> str:
    if note < seuil_neg:
        return "négatif"
    if note < seuil_pos:
        return "neutre"
    return "positif"

## Appel de l'API sur les 30 avis

In [2]:
with open(CSV_PATH, encoding="utf-8") as f:
    avis = list(csv.DictReader(f))

lignes = []
for row in avis:
    resp = requests.post(API_URL, json={"text": row["texte"]}, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    note = note_attendue(data["scores_etoiles"])
    lignes.append(
        {
            "id": row["id"],
            "texte": row["texte"],
            "vrai": row["sentiment_attendu"],
            "note_attendue": round(note, 2),
            "predit_actuel": data["sentiment"],
            "predit_alt": convertir(note, SEUIL_NEG_ALT, SEUIL_POS_ALT),
        }
    )

df = pd.DataFrame(lignes)
df

,id,texte,vrai,note_attendue,predit_actuel,predit_alt
0,1,"Séjour parfait ! Piscine magnifique, personnel...",positif,4.78,positif,positif
1,2,"Chambre sale à notre arrivée, cheveux dans la ...",négatif,1.22,négatif,négatif
2,3,Bravo pour l'insonorisation : j'ai pu suivre e...,négatif,4.70,positif,positif
3,4,"Rien à redire, je ne m'attendais pas à un accu...",positif,4.60,positif,positif
4,5,Hôtel correct pour une nuit d'étape. Chambre p...,neutre,2.98,neutre,neutre
5,6,Le spa n'est pas du tout à la hauteur de ce qu...,négatif,1.59,négatif,négatif
6,7,"Très bien situé, à deux pas de la gare. Person...",positif,4.48,positif,positif
7,8,Nous venons depuis cinq ans et c'est clairemen...,négatif,1.95,négatif,négatif
8,9,"Chambre spacieuse et lumineuse, mais le wifi é...",neutre,3.32,neutre,neutre
9,10,"Réservation au nom de M. et Mme Lefèvre, chamb...",négatif,1.88,négatif,négatif


## Matrices de confusion

In [3]:
def matrice_confusion(df, colonne_predite):
    m = pd.crosstab(df["vrai"], df[colonne_predite]).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
    m.index.name = "vrai \\ prédit"
    return m


def resumer(df, colonne_predite, titre):
    n = len(df)
    bons = (df["vrai"] == df[colonne_predite]).sum()
    neg = df["vrai"] == "négatif"
    rates = (neg & (df[colonne_predite] != "négatif")).sum()
    print(f"{titre} : {bons}/{n} bien classés ({bons/n:.1%}) — {rates}/{neg.sum()} négatifs ratés")
    return bons, rates


m_actuel = matrice_confusion(df, "predit_actuel")
m_alt = matrice_confusion(df, "predit_alt")

print(f"Conversion ACTUELLE (seuils {SEUIL_NEG_ACTUEL}/{SEUIL_POS_ACTUEL})")
display(m_actuel)
bons_actuel, rates_actuel = resumer(df, "predit_actuel", "Actuelle")

print(f"\nConversion ALTERNATIVE (seuils {SEUIL_NEG_ALT}/{SEUIL_POS_ALT})")
display(m_alt)
bons_alt, rates_alt = resumer(df, "predit_alt", "Alternative")

Conversion ACTUELLE (seuils 2.8/3.6)


predit_actuel,négatif,neutre,positif
vrai \ prédit,,,
négatif,10,0,2
neutre,0,7,1
positif,1,0,9


Actuelle : 26/30 bien classés (86.7%) — 2/12 négatifs ratés

Conversion ALTERNATIVE (seuils 2.5/3.5)


predit_alt,négatif,neutre,positif
vrai \ prédit,,,
négatif,10,0,2
neutre,0,6,2
positif,1,0,9


Alternative : 25/30 bien classés (83.3%) — 2/12 négatifs ratés


## Comparaison

In [ ]:
if rates_actuel < rates_alt:
    print("=> La conversion ACTUELLE fait moins d'erreurs graves (négatifs ratés).")
elif rates_alt < rates_actuel:
    print("=> La conversion ALTERNATIVE fait moins d'erreurs graves (négatifs ratés).")
else:
    print("=> Les deux conversions font autant d'erreurs graves (négatifs ratés).")

print(f"Bons classements — actuelle: {bons_actuel}/{len(df)}, alternative: {bons_alt}/{len(df)}")

=> Les deux conversions font autant d'erreurs graves (négatifs ratés).
Bons classements — actuelle: 26/30, alternative: 25/30


: 